# 🚗 Huấn Luyện Chuẩn Hóa YOLO11m Trên Kaggle (UA-DETRAC -> YOLO)

Notebook này được tối ưu đặc thù cho nền tảng **Kaggle Notebooks (GPU P100 hoặc GPU T4 x 2)**.

> ⚠️ **CÀI ĐẶT BẮT BUỘC TRÊN KAGGLE TRƯỚC KHI CHẠY**:
> 1. Ở thanh menu bên phải (**Notebook settings**), mục **Accelerator** chọn: **GPU T4 x 2** hoặc **GPU P100**.
> 2. Mục **Internet**: Chuyển sang **Internet On** (bắt buộc để tải `yolo11m.pt` và cài đặt `ultralytics`).

--- 
### 🎯 Các Vấn Đề Được Khắc Phục Triệt Để:
1. **Tự động chuyển đổi XML -> YOLO**: Đọc trực tiếp từ bộ dữ liệu gốc `bratjay/ua-detrac-orig`, tự sinh nhãn YOLO, dùng symlink tiết kiệm 100% dung lượng ổ cứng Kaggle.
2. **Chống mất nhận diện (Catastrophic Forgetting)**: Sử dụng `freeze=10` để khóa toàn bộ 10 tầng đầu của **Backbone** (0-9: Conv, C3k2, SPPF), bảo tồn tuyệt đối bộ lọc nhận diện xe tổng quát từ pre-trained COCO dataset.
3. **Chống Overfitting**: Tích hợp `patience=5` (Early Stopping) kết hợp bộ điều chỉnh tốc độ học `cos_lr=True` (Cosine Annealing) từ `lr0=0.001` xuống `1e-5`.
4. **Tối ưu Bounding Box**: `close_mosaic=5` tắt mosaic 5 epochs cuối giúp khung bao xe vuông vắn, không bị méo lệch.
5. **Tự động gom file xuất ra `/kaggle/working/`**: Dễ dàng tải trực tiếp `best.pt` và file nén zip về máy tính.


### 1. Kiểm tra Môi Trường Kaggle & Cài Đặt Thư Viện


In [ ]:
# 1. Kiểm tra GPU được cấp phát trên Kaggle
!nvidia-smi

# 2. Cài đặt Ultralytics và các thư viện cần thiết
!pip install -q ultralytics pyyaml matplotlib opencv-python-headless


### 2. Tự Động Chuyển Đổi Dữ Liệu UA-DETRAC (XML -> YOLO) & Khởi Tạo `data.yaml`
Cell này sẽ tự động đọc các file XML trong dataset `bratjay/ua-detrac-orig`, chuyển đổi nhãn sang chuẩn YOLO và tạo liên kết ảnh (Symlink) vào `/kaggle/working/data/` (chỉ mất ~1-2 phút, không tốn dung lượng ổ đĩa).


In [ ]:
import os
import shutil
import random
import xml.etree.ElementTree as ET
import yaml
from pathlib import Path

print("=" * 70)
print("🚀 TỰ ĐỘNG CHUYỂN ĐỔI UA-DETRAC SANG CHUẨN YOLO & KHỞI TẠO DATASET")
print("=" * 70)

# 1. Tự động định vị thư mục dataset UA-DETRAC trên Kaggle
input_root = Path('/kaggle/input')
base_detrac = None
for candidate in [Path('/kaggle/input/datasets/bratjay/ua-detrac-orig'), Path('/kaggle/input/ua-detrac-orig')]:
    if candidate.exists():
        base_detrac = candidate
        break

if base_detrac is None:
    # Quét tìm kiếm thư mục chứa DETRAC-Images
    for r, d, f in os.walk(input_root):
        if 'DETRAC-Images' in d:
            base_detrac = Path(r)
            break

print(f"📁 Thư mục gốc UA-DETRAC tìm thấy: {base_detrac}")

# Định vị thư mục ảnh
images_root = None
for cand in [base_detrac / 'DETRAC-Images' / 'DETRAC-Images', base_detrac / 'DETRAC-Images']:
    if cand.exists() and any(cand.iterdir()):
        images_root = cand
        break

# Định vị thư mục XML
train_xml_dir = None
for cand in [base_detrac / 'DETRAC-Train-Annotations-XML' / 'DETRAC-Train-Annotations-XML', base_detrac / 'DETRAC-Train-Annotations-XML']:
    if cand.exists() and any(cand.glob('*.xml')):
        train_xml_dir = cand
        break

print(f"📁 Thư mục ảnh nguồn : {images_root}")
print(f"📁 Thư mục XML nhãn  : {train_xml_dir}")

if not images_root or not train_xml_dir:
    raise FileNotFoundError("Chưa tìm thấy đủ thư mục DETRAC-Images hoặc DETRAC-Train-Annotations-XML!")

# 2. Cấu hình chuyển đổi
OUTPUT_DIR = Path('/kaggle/working/data')
IMG_WIDTH = 960
IMG_HEIGHT = 540
VAL_RATIO = 0.2        # 20% sequences cho Validation
FRAME_STEP = 5         # Lấy mẫu 1 frame mỗi 5 frame (giảm từ 82k ảnh xuống ~16k ảnh chuẩn)
RANDOM_SEED = 42

def parse_ignored_regions(sequence_root):
    ignored = []
    ignored_elem = sequence_root.find("ignored_region")
    if ignored_elem is not None:
        for box in ignored_elem.findall("box"):
            left = float(box.get("left"))
            top = float(box.get("top"))
            width = float(box.get("width"))
            height = float(box.get("height"))
            ignored.append((left, top, left + width, top + height))
    return ignored

def compute_iou(box_a, box_b):
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def is_in_ignored(box, ignored_list, iou_thresh=0.5):
    for ign in ignored_list:
        if compute_iou(box, ign) > iou_thresh:
            return True
    return False

def convert_to_yolo(left, top, width, height, img_w=IMG_WIDTH, img_h=IMG_HEIGHT):
    xc = max(0.0, min(1.0, (left + width / 2) / img_w))
    yc = max(0.0, min(1.0, (top + height / 2) / img_h))
    w = max(0.0, min(1.0, width / img_w))
    h = max(0.0, min(1.0, height / img_h))
    return xc, yc, w, h

# 3. Chia tập theo Sequence (Sequence-level split chống rò rỉ dữ liệu)
xml_files = sorted(list(train_xml_dir.glob('*.xml')))
seq_names = [f.stem for f in xml_files]
random.seed(RANDOM_SEED)
random.shuffle(seq_names)

n_val = max(1, int(len(seq_names) * VAL_RATIO))
val_seqs = set(seq_names[:n_val])
train_seqs = set(seq_names[n_val:])

print(f"\n📊 Phân chia Sequence: {len(train_seqs)} Train sequences | {len(val_seqs)} Val sequences")

for split in ['train', 'val']:
    (OUTPUT_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

# 4. Tiến hành chuyển đổi nhãn & tạo liên kết ảnh mềm (Symlink)
stats = {'train_frames': 0, 'train_boxes': 0, 'val_frames': 0, 'val_boxes': 0}

print("\n⏳ Đang chuyển đổi annotations XML sang YOLO format...")
for idx, xml_path in enumerate(xml_files):
    seq_name = xml_path.stem
    split = 'val' if seq_name in val_seqs else 'train'
    out_img_dir = OUTPUT_DIR / split / 'images'
    out_lbl_dir = OUTPUT_DIR / split / 'labels'
    
    seq_img_dir = images_root / seq_name
    if not seq_img_dir.exists():
        sub_search = list(images_root.rglob(seq_name))
        if sub_search:
            seq_img_dir = sub_search[0]
        else:
            continue
            
    tree = ET.parse(xml_path)
    root = tree.getroot()
    ignored_regions = parse_ignored_regions(root)
    
    frames = root.findall("frame")
    for frame in frames:
        frame_num = int(frame.get("num"))
        if frame_num % FRAME_STEP != 0:
            continue
            
        img_name = f"img{frame_num:05d}.jpg"
        src_img = seq_img_dir / img_name
        if not src_img.exists():
            continue
            
        unique_id = f"{seq_name}_{img_name.replace('.jpg', '')}"
        dst_img = out_img_dir / f"{unique_id}.jpg"
        dst_lbl = out_lbl_dir / f"{unique_id}.txt"
        
        if not dst_img.exists():
            try:
                os.symlink(src_img, dst_img)
            except OSError:
                shutil.copy2(src_img, dst_img)
                
        lines = []
        target_list = frame.find("target_list")
        if target_list is not None:
            for target in target_list.findall("target"):
                box = target.find("box")
                if box is None:
                    continue
                left = float(box.get("left"))
                top = float(box.get("top"))
                width = float(box.get("width"))
                height = float(box.get("height"))
                
                if width < 10 or height < 10:
                    continue
                if is_in_ignored((left, top, left + width, top + height), ignored_regions):
                    continue
                    
                xc, yc, w, h = convert_to_yolo(left, top, width, height)
                lines.append(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
                stats[f"{split}_boxes"] += 1
                
        with open(dst_lbl, 'w', encoding='utf-8') as f:
            f.write("\n".join(lines))
        stats[f"{split}_frames"] += 1

print("\n" + "=" * 70)
print("✅ HOÀN TẤT CHUYỂN ĐỔI DATASET UA-DETRAC SANG YOLO!")
print(f"• Tập Train: {stats['train_frames']:,} ảnh | {stats['train_boxes']:,} nhãn xe")
print(f"• Tập Val  : {stats['val_frames']:,} ảnh | {stats['val_boxes']:,} nhãn xe")
print("=" * 70)

# 5. Khởi tạo file data.yaml chuẩn
data_yaml_cfg = {
    'path': '/kaggle/working/data',
    'train': 'train/images',
    'val': 'val/images',
    'test': 'val/images',
    'nc': 1,
    'names': {0: 'vehicle'}
}

yaml_file = '/kaggle/working/data.yaml'
with open(yaml_file, 'w', encoding='utf-8') as f:
    yaml.dump(data_yaml_cfg, f, default_flow_style=False, sort_keys=False)

print(f"\n📄 Đã sinh file cấu hình chuẩn: {yaml_file}")
with open(yaml_file, 'r') as f:
    print(f.read())
print("👉 Bạn đã có thể chạy tiếp Cell 3 và Cell 4 để bắt đầu train mô hình!")


### 3. Tải Mô Hình `YOLO11m` & Xác Minh Cơ Chế Khóa Tầng (Backbone Freezing)

YOLO11 gồm tổng cộng 24 tầng (0 đến 23):
- **Tầng 0 -> 9 (Backbone)**: Được **KHÓA** (`freeze=10`) để bảo vệ toàn bộ tri thức xe đa góc nhìn từ tập dữ liệu COCO.
- **Tầng 10 -> 22 (Neck)**: Được **MỞ** để kết hợp đặc trưng tỉ lệ cho camera đường phố.
- **Tầng 23 (Detection Head)**: Được **MỞ** để chuyên môn hóa dự đoán bounding box cho nhãn `vehicle`.


In [ ]:
from ultralytics import YOLO

# Khởi tạo mô hình pre-trained YOLO11m (Medium)
model = YOLO('yolo11m.pt')

total_layers = len(list(model.model.model))
total_params = sum(p.numel() for p in model.model.parameters())
print("=" * 60)
print("✅ Khởi tạo mô hình YOLO11m thành công!")
print(f"• Tổng số tầng kiến trúc : {total_layers} tầng (0 đến {total_layers - 1})")
print(f"• Tổng số lượng tham số  : {total_params:,} parameters (~20.1M)")
print("• Chiến lược Fine-tuning : freeze=10 (Khóa cứng 10 tầng Backbone 0-9)")
print("=" * 60)


### 4. Tiến Hành Huấn Luyện (Training Với Hyperparameters Chuẩn Hóa Trên Kaggle)

| Tham Số | Giá Trị | Mục Đích |
| :--- | :--- | :--- |
| **data** | `/kaggle/working/data.yaml` | Đường dẫn file cấu hình vừa sinh |
| **freeze** | `10` | Khóa 10 tầng đầu của Backbone, chống Catastrophic Forgetting |
| **epochs** | `30` | Số epoch tối đa |
| **patience** | `7` | Dừng sớm nếu val/loss hoặc mAP không cải thiện (chống Overfitting) |
| **batch** | `16` | Tối ưu cho mô hình Medium trên GPU Kaggle 16GB (tránh tràn VRAM CUDA OOM) |
| **imgsz** | `640` | Kích thước ảnh chuẩn hóa |
| **lr0** | `0.001` | Learning rate khởi đầu nhẹ nhàng cho fine-tuning |
| **cos_lr** | `True` | Giảm tốc độ học mượt mà theo hàm Cosine |
| **close_mosaic**| `5` | Tắt ghép 4 ảnh ở 5 epochs cuối để bounding box chuẩn xác |
| **device** | `0` | Sử dụng GPU chính trên Kaggle (tránh lỗi multiprocessing DDP khi chạy interactive) |


In [ ]:
# Thực thi quá trình huấn luyện
results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=25,
    patience=5,              # Tự động ngắt khi phát hiện dấu hiệu Overfitting
    batch=16,                # Batch size 16 an toàn cho model Medium (16GB VRAM)
    imgsz=640,
    freeze=10,               # Khóa 10 tầng backbone (0-9)
    lr0=0.001,               # LR ban đầu nhẹ nhàng
    lrf=0.01,                # LR cuối cùng = 1e-5
    cos_lr=True,             # Giảm LR theo Cosine Annealing
    warmup_epochs=2,         # Làm ấm mô hình 2 epochs đầu
    close_mosaic=5,          # Tắt mosaic trong 5 epochs cuối
    project='/kaggle/working/runs',
    name='yolo11m_vehicle',
    workers=4,
    device=0,                # GPU 0 trên Kaggle
    verbose=True,
    plots=True
)

print("\n" + "=" * 60)
print("🎉 HUẤN LUYỆN HOÀN TẤT THÀNH CÔNG!")
print("Trọng số tốt nhất (Best Checkpoint): /kaggle/working/runs/yolo11m_vehicle/weights/best.pt")
print("=" * 60)


### 5. Trực Quan Hóa Đồ Thị Huấn Luyện (Loss Curves, mAP & Confusion Matrix)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

run_dir = '/kaggle/working/runs/yolo11m_vehicle'

fig, axes = plt.subplots(1, 2, figsize=(22, 9))

# 1. Đồ thị Loss & mAP
res_img_path = os.path.join(run_dir, 'results.png')
if os.path.exists(res_img_path):
    axes[0].imshow(mpimg.imread(res_img_path))
    axes[0].set_title('Biểu đồ Huấn Luyện: Loss, Precision, Recall & mAP', fontsize=14, fontweight='bold')
    axes[0].axis('off')
else:
    axes[0].text(0.5, 0.5, 'results.png chưa hoàn tất', ha='center')

# 2. Ma trận nhầm lẫn (Confusion Matrix)
cm_path = os.path.join(run_dir, 'confusion_matrix_normalized.png')
if not os.path.exists(cm_path):
    cm_path = os.path.join(run_dir, 'confusion_matrix.png')
if os.path.exists(cm_path):
    axes[1].imshow(mpimg.imread(cm_path))
    axes[1].set_title('Normalized Confusion Matrix', fontsize=14, fontweight='bold')
    axes[1].axis('off')
else:
    axes[1].text(0.5, 0.5, 'confusion_matrix.png chưa hoàn tất', ha='center')

plt.tight_layout()
plt.show()


### 6. Đánh Giá Độ Chính Xác Định Lượng Trên Tập Validation


In [ ]:
from ultralytics import YOLO

# Nạp model vừa train từ best checkpoint
best_model = YOLO('/kaggle/working/runs/yolo11m_vehicle/weights/best.pt')

# Đánh giá validation
metrics = best_model.val(data='/kaggle/working/data.yaml', split='val')
print("\n" + "=" * 50)
print("📊 THÔNG SỐ ĐÁNH GIÁ TRÊN TẬP VALIDATION:")
print(f"• mAP@50     : {metrics.box.map50:.4f} ({metrics.box.map50*100:.2f}%)")
print(f"• mAP@50-95  : {metrics.box.map:.4f} ({metrics.box.map*100:.2f}%)")
print(f"• Precision  : {metrics.box.mp:.4f}")
print(f"• Recall     : {metrics.box.mr:.4f}")
print("=" * 50)


### 7. Xuất Trọng Số Ra `/kaggle/working/` & Tạo Link Tải Trực Tiếp
Trên Kaggle, bất kỳ file nào nằm trong thư mục `/kaggle/working/` đều có thể tải trực tiếp từ panel **Output** ở góc dưới bên phải hoặc bấm vào link tải được tạo dưới đây.


In [ ]:
import os
import shutil
from IPython.display import FileLink

# 1. Sao chép best.pt và last.pt trực tiếp ra /kaggle/working/ để tiện tải riêng lẻ
shutil.copy2('/kaggle/working/runs/yolo11m_vehicle/weights/best.pt', '/kaggle/working/best.pt')
if os.path.exists('/kaggle/working/runs/yolo11m_vehicle/weights/last.pt'):
    shutil.copy2('/kaggle/working/runs/yolo11m_vehicle/weights/last.pt', '/kaggle/working/last.pt')

# 2. Gom toàn bộ model + báo cáo đồ thị thành file zip
export_dir = '/kaggle/working/yolo11m_export'
os.makedirs(f"{export_dir}/weights", exist_ok=True)
shutil.copy2('/kaggle/working/best.pt', f"{export_dir}/weights/best.pt")

for report_file in ['results.png', 'results.csv', 'confusion_matrix.png', 'args.yaml']:
    src = f'/kaggle/working/runs/yolo11m_vehicle/{report_file}'
    if os.path.exists(src):
        shutil.copy2(src, f"{export_dir}/{report_file}")

zip_path = '/kaggle/working/output_runs_yolo11m'
shutil.make_archive(zip_path, 'zip', export_dir)
print(f"✅ Đã tạo file nén: {zip_path}.zip")

# 3. Hiển thị link tải trực tiếp trong Kaggle UI
print("\n👇 BẤM VÀO ĐÂY ĐỂ TẢI VỀ MÁY TÍNH:")
display(FileLink('best.pt'))
display(FileLink('output_runs_yolo11m.zip'))
print("\n👉 Hoặc bạn có thể tải từ panel 'Output' ở góc phải giao diện Kaggle!")
